# Delta Lake MERGE Assignment - Phase 1
## Data Creation and Cleaning using Pandas

### Objective
In this notebook, we will:

- Create a Customer Master dataset
- Create an Incremental dataset
- Perform data exploration
- Handle missing values
- Remove duplicate records
- Correct data types
- Save cleaned datasets as CSV files

These cleaned CSV files will later be uploaded to **Azure Databricks** for implementing **Delta Lake MERGE (SCD Type 1 & Type 2)**.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

## Step 1 : Create Customer Master Dataset

The Customer Master dataset represents the existing customer records stored in the company database.

In [ ]:
customer_ids = list(range(1001, 1101))

names = [
    "Aarav","Vivaan","Aditya","Vihaan","Arjun","Sai","Krishna","Ishaan",
    "Ananya","Diya","Aadhya","Sara","Riya","Priya","Sneha","Pooja",
    "Rahul","Rohan","Amit","Kunal","Neha","Meera","Nikhil","Kiran"
]

cities = [
    "Pune","Mumbai","Nashik","Nagpur","Delhi",
    "Bengaluru","Hyderabad","Chennai","Jaipur","Indore"
]

master_df = pd.DataFrame({
    "customer_id": customer_ids,
    "customer_name": np.random.choice(names,100),
    "city": np.random.choice(cities,100),
    "age": np.random.randint(20,60,100),
    "product": np.random.choice(
        ["Laptop","Phone","Tablet","Headphones","Keyboard","Mouse"],100),
    "quantity": np.random.randint(1,6,100),
    "price": np.random.randint(500,50000,100)
})

master_df["total_amount"] = master_df["quantity"] * master_df["price"]

master_df.head()

,customer_id,customer_name,city,age,product,quantity,price,total_amount
0,1001,Krishna,Chennai,38,Phone,5,49549,247745
1,1002,Kunal,Chennai,21,Keyboard,4,15805,63220
2,1003,Sneha,Nashik,45,Mouse,5,10317,51585
3,1004,Aadhya,Pune,51,Laptop,4,11761,47044
4,1005,Ishaan,Chennai,25,Headphones,3,917,2751


# Introduce Data Quality Issues

Real-world data is rarely perfect.

To simulate realistic scenarios, we intentionally introduce:
- Missing values
- Duplicate records

In [ ]:
master_df.loc[5,"city"] = np.nan
master_df.loc[15,"customer_name"] = np.nan
master_df.loc[25,"price"] = np.nan

duplicate_rows = master_df.iloc[[10,20]]

master_df = pd.concat([master_df,duplicate_rows],ignore_index=True)

master_df

,customer_id,customer_name,city,age,product,quantity,price,total_amount
0,1001,Krishna,Chennai,38,Phone,5,49549.0,247745
1,1002,Kunal,Chennai,21,Keyboard,4,15805.0,63220
2,1003,Sneha,Nashik,45,Mouse,5,10317.0,51585
3,1004,Aadhya,Pune,51,Laptop,4,11761.0,47044
4,1005,Ishaan,Chennai,25,Headphones,3,917.0,2751
...,...,...,...,...,...,...,...,...
97,1098,Kiran,Delhi,27,Headphones,1,47232.0,47232
98,1099,Aarav,Pune,46,Headphones,2,39581.0,79162
99,1100,Sara,Pune,46,Headphones,5,28069.0,140345
100,1011,Aadhya,Hyderabad,43,Headphones,3,39013.0,117039


# Generate Incremental Dataset

This dataset simulates newly received customer records.

It contains:
- Existing customers with updated information
- Completely new customers

In [ ]:
increment_df = pd.DataFrame({
    "customer_id":[1003,1012,1025,1040,1055,1101,1102,1103,1104,1105],
    "customer_name":[
        "Aditya","Sneha","Rahul","Meera","Aarav",
        "Yash","Tanvi","Om","Siya","Ved"
    ],
    "city":[
        "Pune","Mumbai","Delhi","Nagpur","Hyderabad",
        "Surat","Lucknow","Pune","Mumbai","Goa"
    ],
    "age":[31,29,40,36,27,25,24,28,30,26],
    "product":[
        "Laptop","Phone","Tablet","Keyboard","Mouse",
        "Laptop","Phone","Tablet","Laptop","Mouse"
    ],
    "quantity":[2,1,3,1,4,2,1,2,3,1],
    "price":[60000,30000,25000,4000,1500,45000,22000,18000,50000,2000]
})

increment_df["total_amount"] = increment_df["quantity"] * increment_df["price"]

increment_df

,customer_id,customer_name,city,age,product,quantity,price,total_amount
0,1003,Aditya,Pune,31,Laptop,2,60000,120000
1,1012,Sneha,Mumbai,29,Phone,1,30000,30000
2,1025,Rahul,Delhi,40,Tablet,3,25000,75000
3,1040,Meera,Nagpur,36,Keyboard,1,4000,4000
4,1055,Aarav,Hyderabad,27,Mouse,4,1500,6000
5,1101,Yash,Surat,25,Laptop,2,45000,90000
6,1102,Tanvi,Lucknow,24,Phone,1,22000,22000
7,1103,Om,Pune,28,Tablet,2,18000,36000
8,1104,Siya,Mumbai,30,Laptop,3,50000,150000
9,1105,Ved,Goa,26,Mouse,1,2000,2000


# Save the Datasets

The generated datasets are saved as CSV files and will be uploaded to Azure Databricks for further processing.

In [ ]:
master_df.to_csv("customer_master.csv",index=False)

increment_df.to_csv("customer_incremental.csv",index=False)

print("Datasets saved successfully.")

Datasets saved successfully.


In [ ]:
print(master_df.shape)
print(increment_df.shape)

(102, 8)
(10, 8)


In [ ]:
master_df.sample(10)

,customer_id,customer_name,city,age,product,quantity,price,total_amount
100,1011,Aadhya,Hyderabad,43,Headphones,3,39013.0,117039
47,1048,Priya,Hyderabad,45,Tablet,2,1228.0,2456
41,1042,Krishna,Nagpur,42,Phone,4,21852.0,87408
6,1007,Krishna,Nashik,23,Keyboard,5,20858.0,104290
21,1022,Sara,Delhi,48,Tablet,5,19388.0,96940
39,1040,Arjun,Hyderabad,47,Phone,1,47820.0,47820
42,1043,Neha,Nagpur,50,Tablet,5,46852.0,234260
81,1082,Nikhil,Pune,23,Mouse,5,38602.0,193010
1,1002,Kunal,Chennai,21,Keyboard,4,15805.0,63220
98,1099,Aarav,Pune,46,Headphones,2,39581.0,79162


In [ ]:
increment_df

,customer_id,customer_name,city,age,product,quantity,price,total_amount
0,1003,Aditya,Pune,31,Laptop,2,60000,120000
1,1012,Sneha,Mumbai,29,Phone,1,30000,30000
2,1025,Rahul,Delhi,40,Tablet,3,25000,75000
3,1040,Meera,Nagpur,36,Keyboard,1,4000,4000
4,1055,Aarav,Hyderabad,27,Mouse,4,1500,6000
5,1101,Yash,Surat,25,Laptop,2,45000,90000
6,1102,Tanvi,Lucknow,24,Phone,1,22000,22000
7,1103,Om,Pune,28,Tablet,2,18000,36000
8,1104,Siya,Mumbai,30,Laptop,3,50000,150000
9,1105,Ved,Goa,26,Mouse,1,2000,2000


# Data Cleaning using Pandas

Before using the datasets in Azure Databricks, we clean the data by:

- Exploring the dataset
- Checking data types
- Handling missing values
- Removing duplicate records
- Validating data
- Saving the cleaned datasets

In [ ]:
master_df = pd.read_csv("customer_master.csv")
increment_df = pd.read_csv("customer_incremental.csv")

In [ ]:
master_df.head()

,customer_id,customer_name,city,age,product,quantity,price,total_amount
0,1001,Krishna,Chennai,38,Phone,5,49549.0,247745
1,1002,Kunal,Chennai,21,Keyboard,4,15805.0,63220
2,1003,Sneha,Nashik,45,Mouse,5,10317.0,51585
3,1004,Aadhya,Pune,51,Laptop,4,11761.0,47044
4,1005,Ishaan,Chennai,25,Headphones,3,917.0,2751


In [ ]:
master_df.tail()

,customer_id,customer_name,city,age,product,quantity,price,total_amount
97,1098,Kiran,Delhi,27,Headphones,1,47232.0,47232
98,1099,Aarav,Pune,46,Headphones,2,39581.0,79162
99,1100,Sara,Pune,46,Headphones,5,28069.0,140345
100,1011,Aadhya,Hyderabad,43,Headphones,3,39013.0,117039
101,1021,Kiran,Chennai,35,Mouse,4,39256.0,157024


In [ ]:
master_df.shape

(102, 8)

In [ ]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102 entries, 0 to 101
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   customer_id    102 non-null    int64  
 1   customer_name  101 non-null    object 
 2   city           101 non-null    object 
 3   age            102 non-null    int64  
 4   product        102 non-null    object 
 5   quantity       102 non-null    int64  
 6   price          101 non-null    float64
 7   total_amount   102 non-null    int64  
dtypes: float64(1), int64(4), object(3)
memory usage: 6.5+ KB


In [ ]:
master_df.describe()

,customer_id,age,quantity,price,total_amount
count,102.000000,102.000000,102.000000,101.000000,102.000000
mean,1049.823529,40.000000,3.186275,25147.019802,79877.754902
std,29.130778,11.704277,1.440093,14784.565424,63803.663728
min,1001.000000,20.000000,1.000000,912.000000,2396.000000
25%,1024.250000,30.000000,2.000000,11761.000000,28401.000000
50%,1049.500000,39.500000,3.000000,24076.000000,61650.000000
75%,1074.750000,51.000000,4.750000,38341.000000,119897.500000
max,1100.000000,59.000000,5.000000,49549.000000,247745.000000


In [ ]:
master_df.isnull().sum()

,0
customer_id,0
customer_name,1
city,1
age,0
product,0
quantity,0
price,1
total_amount,0


In [ ]:
master_df["city"] = master_df["city"].fillna("Unknown")

In [ ]:
master_df["customer_name"] = master_df["customer_name"].fillna("Not Available")

In [ ]:
master_df["price"] = master_df["price"].fillna(master_df["price"].median())

In [ ]:
master_df.isnull().sum()

,0
customer_id,0
customer_name,0
city,0
age,0
product,0
quantity,0
price,0
total_amount,0


In [ ]:
master_df.duplicated().sum()

np.int64(2)

In [ ]:
master_df = master_df.drop_duplicates()

In [ ]:
master_df.duplicated().sum()

np.int64(0)

In [ ]:
master_df.dtypes


,0
customer_id,int64
customer_name,object
city,object
age,int64
product,object
quantity,int64
price,float64
total_amount,int64


In [ ]:
master_df["total_amount"] = master_df["price"] * master_df["quantity"]

In [ ]:
master_df.head()

,customer_id,customer_name,city,age,product,quantity,price,total_amount
0,1001,Krishna,Chennai,38,Phone,5,49549.0,247745.0
1,1002,Kunal,Chennai,21,Keyboard,4,15805.0,63220.0
2,1003,Sneha,Nashik,45,Mouse,5,10317.0,51585.0
3,1004,Aadhya,Pune,51,Laptop,4,11761.0,47044.0
4,1005,Ishaan,Chennai,25,Headphones,3,917.0,2751.0


In [ ]:
master_df.shape

(100, 8)

In [ ]:
master_df.to_csv("customer_master_cleaned.csv", index=False)

increment_df.to_csv("customer_incremental_cleaned.csv", index=False)

In [ ]:
from google.colab import files

files.download("customer_master_cleaned.csv")
files.download("customer_incremental_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>